<a href="https://colab.research.google.com/github/14-169-RaisyaSyifaSaleh/Kelompok11_Studio2/blob/main/Studio2_123140169_123140212.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kompetisi Klasifikasi Sampah**

---

### **1. Anggota Kelompok**
* **Anggota 1:** [Raisya Syifa Saleh] - [123140169]
* **Anggota 2:** [Diva Amelia Saputri] - [123140212]

---

### **2. Deskripsi Tugas**
Bangun model klasifikasi citra yang menerima satu foto sampah dan memprediksi salah satu dari lima kelas di bawah. Anda bebas memilih pendekatan: CNN from scratch seperti di Pertemuan 4, arsitektur modern dari Pertemuan 5, atau fine-tuning backbone pretrained ImageNet sesuai materi Studio 2. Data tambahan dari luar juga diperbolehkan.

Kelompok tidak pernah melihat validation set. Asisten dosen yang menjalankan model (model.pth) kelompok Anda pada validation set menggunakan skrip evaluate.py yang sama persis dengan yang ada di starter kit, lalu mencatat skornya di Google Sheets skor kelompok.

---



In [1]:
import os
import sys
import torch
import torchvision
from PIL import Image

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))
    print("Memori GPU:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("GPU belum aktif. Periksa Runtime > Change runtime type.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU tersedia: True
Nama GPU: Tesla T4
Memori GPU: 14.56 GB


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os

dataset_folder = "/content/drive/MyDrive/Studio2_Kelompok11"

print("Lokasi dataset:", dataset_folder)
print("Folder ditemukan:", os.path.exists(dataset_folder))

if os.path.exists(dataset_folder):
    print("\nIsi folder:")
    for item in os.listdir(dataset_folder):
        print("-", item)
else:
    print("Folder belum ditemukan. Periksa lokasi Google Drive.")

Lokasi dataset: /content/drive/MyDrive/Studio2_Kelompok11
Folder ditemukan: True

Isi folder:
- dataset_mahasiswa


In [4]:
import os

dataset_folder = "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"

print("Lokasi dataset:", dataset_folder)
print("Folder ditemukan:", os.path.exists(dataset_folder))

if os.path.exists(dataset_folder):
    print("\nIsi folder dataset_mahasiswa:")
    for item in os.listdir(dataset_folder):
        print("-", item)
else:
    print("Folder tidak ditemukan.")

Lokasi dataset: /content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa
Folder ditemukan: True

Isi folder dataset_mahasiswa:
- train_sources.csv
- BACA_SAYA.txt
- train
- starter_kit
- local_split
- outputs


In [5]:
from pathlib import Path

dataset_folder = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)

train_dir = dataset_folder / "train"

expected_classes = [
    "botol_plastik",
    "gelas_plastik",
    "kertas_kardus",
    "saset_kemasan",
    "styrofoam"
]

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Folder train ditemukan:", train_dir.exists())
print("\nIsi folder train:")

if train_dir.exists():
    for item in sorted(train_dir.iterdir()):
        print("-", item.name)

    print("\nJumlah gambar per kelas:")

    for class_name in expected_classes:
        class_dir = train_dir / class_name

        if class_dir.exists():
            images = [
                file for file in class_dir.rglob("*")
                if file.is_file()
                and file.suffix.lower() in image_extensions
            ]

            print(f"{class_name}: {len(images)} gambar")
        else:
            print(f"{class_name}: FOLDER TIDAK DITEMUKAN")
else:
    print("Folder train tidak ditemukan. Periksa kembali lokasi dataset.")

Folder train ditemukan: True

Isi folder train:
- botol_plastik
- gelas_plastik
- kertas_kardus
- saset_kemasan
- styrofoam

Jumlah gambar per kelas:
botol_plastik: 523 gambar
gelas_plastik: 537 gambar
kertas_kardus: 365 gambar
saset_kemasan: 673 gambar
styrofoam: 671 gambar


In [6]:
from pathlib import Path

starter_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/"
    "dataset_mahasiswa/starter_kit"
)

print("Starter kit ditemukan:", starter_dir.exists())

if starter_dir.exists():
    print("\nDaftar file starter kit:")

    for item in sorted(starter_dir.rglob("*")):
        if item.is_file():
            print("-", item.relative_to(starter_dir))

Starter kit ditemukan: True

Daftar file starter kit:
- README_TEMPLATE.md
- evaluate.py
- submission/model.py
- train_baseline.py


In [7]:
from pathlib import Path

dataset_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)

files_to_read = [
    dataset_dir / "BACA_SAYA.txt",
    dataset_dir / "starter_kit" / "submission" / "model.py",
    dataset_dir / "starter_kit" / "README_TEMPLATE.md",
]

for file_path in files_to_read:
    print("\n" + "=" * 60)
    print("FILE:", file_path.name)
    print("=" * 60)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File tidak ditemukan:", file_path)



FILE: BACA_SAYA.txt
STUDIO 2 — KOMPETISI KLASIFIKASI SAMPAH
Deep Learning (IF25-40401) · Teknik Informatika ITERA · Ganjil 2026/2027

Halaman kompetisi (aturan, jadwal, penilaian):
  https://informatika-itera.github.io/deep-learning-IF25-40401/studio-2/

Isi:
  train/<kelas>/*.jpg   train set, 5 kelas:
                        0 botol_plastik, 1 gelas_plastik, 2 kertas_kardus,
                        3 saset_kemasan, 4 styrofoam
  train_sources.csv     sumber (URL) setiap gambar
  starter_kit/          evaluate.py (skrip evaluasi resmi), train_baseline.py,
                        submission/model.py (contoh), README_TEMPLATE.md

Mulai cepat:
  cd starter_kit
  python train_baseline.py --data ../train --epochs 8
  python evaluate.py --submission ./submission --data ./my_val

Validation set tidak dibagikan. Asisten dosen menjalankan model.pth kelompok
Anda dengan evaluate.py yang sama pada setiap tenggat preview.

Gambar dikumpulkan dari internet untuk keperluan pendidikan non-komersial.

In [8]:
from pathlib import Path

starter_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/"
    "dataset_mahasiswa/starter_kit"
)

for filename in ["train_baseline.py", "evaluate.py"]:
    file_path = starter_dir / filename

    print("\n" + "=" * 60)
    print(f"FILE: {filename}")
    print("=" * 60)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File tidak ditemukan:", file_path)


FILE: train_baseline.py
"""Baseline sederhana Studio 2: fine-tuning ResNet-18 (pretrained ImageNet).

Titik awal saja — kelompok diharapkan melakukan eksperimen sendiri
(backbone lain, augmentasi, freezing, scheduler, data tambahan, dll.).

    python train_baseline.py --data ./train --epochs 8
    python train_baseline.py --data ./train --epochs 8 --scratch   # tanpa bobot pretrained

Hasil: submission/model.pth + folder my_val/ (split validasi lokal 15%)
       yang bisa langsung diuji dengan:
    python evaluate.py --submission ./submission --data ./my_val
"""
import argparse
import random
import shutil
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from submission.model import get_model, get_transform


def make_local_split(src, dst, frac, seed):
    """Salin 15% gambar per kelas dari train ke my_val/ (sekali saja), sisanya ke my_train/."""
    dst_tr, dst_va = Path(dst) / "

In [9]:
from pathlib import Path
from collections import Counter

dataset_dir = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)
train_dir = dataset_dir / "train"

image_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Format gambar per kelas:\n")

for class_dir in sorted(train_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    extensions = Counter(
        p.suffix.lower()
        for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_extensions
    )

    print(f"{class_dir.name}:")
    for ext, count in sorted(extensions.items()):
        print(f"  {ext}: {count}")

Format gambar per kelas:

botol_plastik:
  .jpg: 523
gelas_plastik:
  .jpg: 537
kertas_kardus:
  .jpg: 365
saset_kemasan:
  .jpg: 673
styrofoam:
  .jpg: 671


In [10]:

from pathlib import Path
from collections import Counter
import random
import shutil

# Lokasi dataset di Google Drive
DATASET_DIR = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)
SOURCE_DIR = DATASET_DIR / "train"

# Lokasi hasil pembagian data
SPLIT_DIR = DATASET_DIR / "local_split"
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"

# Pengaturan pembagian data
SEED = 42
VAL_RATIO = 0.15

CLASSES = [
    "botol_plastik",
    "gelas_plastik",
    "kertas_kardus",
    "saset_kemasan",
    "styrofoam",
]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Pastikan dataset sumber tersedia
assert SOURCE_DIR.exists(), (
    f"Folder dataset tidak ditemukan: {SOURCE_DIR}"
)

# Jangan menimpa pembagian sebelumnya secara tidak sengaja
if SPLIT_DIR.exists():
    # Periksa apakah hasil pembagian sebelumnya sudah lengkap
    if (
        TRAIN_DIR.is_dir()
        and VAL_DIR.is_dir()
        and all((TRAIN_DIR / c).is_dir() for c in CLASSES)
        and all((VAL_DIR / c).is_dir() for c in CLASSES)
    ):
        print("Folder hasil pembagian sudah ada.")
        print("Menggunakan hasil pembagian sebelumnya.")

    else:
        raise FileExistsError(
            f"Folder {SPLIT_DIR} sudah ada, tetapi hasil "
            "pembagian belum lengkap. Periksa folder sebelumnya "
            "sebelum membuat ulang."
        )

else:
    random.seed(SEED)

    # Pastikan semua kelas ditemukan
    for class_name in CLASSES:
        class_dir = SOURCE_DIR / class_name
        assert class_dir.is_dir(), (
            f"Folder kelas tidak ditemukan: {class_dir}"
        )

    # Buat folder hasil pembagian
    for class_name in CLASSES:
        (TRAIN_DIR / class_name).mkdir(parents=True)
        (VAL_DIR / class_name).mkdir(parents=True)

    # Bagi gambar di setiap kelas
    for class_name in CLASSES:
        class_dir = SOURCE_DIR / class_name

        images = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        )

        random.shuffle(images)

        val_count = round(len(images) * VAL_RATIO)
        val_images = images[:val_count]
        train_images = images[val_count:]

        for image_path in train_images:
            shutil.copy2(
                image_path,
                TRAIN_DIR / class_name / image_path.name
            )

        for image_path in val_images:
            shutil.copy2(
                image_path,
                VAL_DIR / class_name / image_path.name
            )

# Tampilkan hasil pembagian
print("HASIL PEMBAGIAN DATASET")
print("-" * 45)

total_train = 0
total_val = 0

for class_name in CLASSES:
    n_train = len(list((TRAIN_DIR / class_name).glob("*")))
    n_val = len(list((VAL_DIR / class_name).glob("*")))

    total_train += n_train
    total_val += n_val

    print(
        f"{class_name:18s} | "
        f"Train: {n_train:4d} | Val: {n_val:3d}"
    )

print("-" * 45)
print(f"Total training   : {total_train}")
print(f"Total validasi   : {total_val}")
print(f"Total keseluruhan: {total_train + total_val}")


Folder hasil pembagian sudah ada.
Menggunakan hasil pembagian sebelumnya.
HASIL PEMBAGIAN DATASET
---------------------------------------------
botol_plastik      | Train:  445 | Val:  78
gelas_plastik      | Train:  456 | Val:  81
kertas_kardus      | Train:  310 | Val:  55
saset_kemasan      | Train:  572 | Val: 101
styrofoam          | Train:  570 | Val: 101
---------------------------------------------
Total training   : 2353
Total validasi   : 416
Total keseluruhan: 2769


In [11]:

from pathlib import Path

DATASET_DIR = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)

for folder in ["train", "val"]:
    split_path = DATASET_DIR / "local_split" / folder
    print(f"{folder}: {split_path.exists()}")

    if split_path.exists():
        for class_dir in sorted(split_path.iterdir()):
            if class_dir.is_dir():
                print(f"  {class_dir.name}: {len(list(class_dir.glob('*')))} file")


train: True
  botol_plastik: 445 file
  gelas_plastik: 456 file
  kertas_kardus: 310 file
  saset_kemasan: 572 file
  styrofoam: 570 file
val: True
  botol_plastik: 78 file
  gelas_plastik: 81 file
  kertas_kardus: 55 file
  saset_kemasan: 101 file
  styrofoam: 101 file


In [12]:

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU belum aktif. Buka Runtime > Change runtime type,")
    print("lalu pilih GPU pada Hardware accelerator.")


PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU tersedia: True
Nama GPU: Tesla T4


In [13]:

from pathlib import Path
import torch
import torchvision
from google.colab import drive

# Hubungkan Google Drive
drive.mount("/content/drive")

# Lokasi dataset yang sudah dibagi menjadi train dan val
DATASET_DIR = Path(
    "/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa"
)
SPLIT_DIR = DATASET_DIR / "local_split"
TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"

# Lokasi penyimpanan hasil training
OUTPUT_DIR = DATASET_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "botol_plastik",
    "gelas_plastik",
    "kertas_kardus",
    "saset_kemasan",
    "styrofoam",
]

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Perangkat:", "GPU" if torch.cuda.is_available() else "CPU")
print("Folder train tersedia:", TRAIN_DIR.exists())
print("Folder val tersedia:", VAL_DIR.exists())
print("Folder hasil:", OUTPUT_DIR)

if not TRAIN_DIR.exists() or not VAL_DIR.exists():
    raise FileNotFoundError(
        "Folder train/val tidak ditemukan. Periksa lokasi dataset."
    )

for kelas in CLASSES:
    jumlah_train = len(list((TRAIN_DIR / kelas).glob("*.jpg")))
    jumlah_val = len(list((VAL_DIR / kelas).glob("*.jpg")))
    print(f"{kelas}: train={jumlah_train}, val={jumlah_val}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Perangkat: GPU
Folder train tersedia: True
Folder val tersedia: True
Folder hasil: /content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa/outputs
botol_plastik: train=445, val=78
gelas_plastik: train=456, val=81
kertas_kardus: train=310, val=55
saset_kemasan: train=572, val=101
styrofoam: train=570, val=101


In [14]:
import os
from pathlib import Path

# 1. Buat folder 'submission' di penyimpanan sementara Colab
sub_dir = Path("submission")
sub_dir.mkdir(parents=True, exist_ok=True)

# 2. Isi kode model.py sesuai instruksi panduan (ResNet-18, 5 Kelas)
model_py_content = """\"\"\"Submission ResNet-18 untuk Studio 2.\"\"\"
import torch.nn as nn
from torchvision import models, transforms

NUM_CLASSES = 5

def get_model():
    # Membuat struktur ResNet-18 tanpa memuat bobot di fungsi ini
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

def get_transform():
    # Transformasi standar validasi/pengujian
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
"""

# Simpan ke submission/model.py
with open(sub_dir / "model.py", "w", encoding="utf-8") as f:
    f.write(model_py_content)

print("✅ Folder dan file 'submission/model.py' berhasil dibuat!")

✅ Folder dan file 'submission/model.py' berhasil dibuat!


In [15]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from pathlib import Path
from PIL import Image

# Import fungsi get_transform dari model.py yang baru dibuat
from submission.model import get_transform

# --- KONFIGURASI GPU & 10 EPOCH ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 10      # Mengubah jumlah epoch menjadi 10
BATCH_SIZE = 16
LR = 3e-4
CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]

# Lokasi Data dari Google Drive Anda
DATASET_DIR = Path("/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa/local_split")
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
SUBMISSION_DIR = Path("submission")

print(f"Perangkat yang digunakan: {DEVICE}")

# 1. Augmentasi Data Training & Transformasi Validasi
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = get_transform()

# 2. Load Dataset
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds = datasets.ImageFolder(VAL_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# 3. Fungsi Perhitungan Macro-F1 kustom
def compute_macro_f1(y_true, y_pred, num_classes=5):
    cm = [[0] * num_classes for _ in range(num_classes)]
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    f1s = []
    for c in range(num_classes):
        tp = cm[c][c]
        fp = sum(cm[r][c] for r in range(num_classes)) - tp
        fn = sum(cm[c]) - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return sum(f1s) / num_classes

# 4. Inisialisasi Model ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

best_f1 = 0.0

print(f"\n--- Memulai Training di {DEVICE.upper()} ---")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # Training Loop
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_ds)

    # Validation Loop
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().tolist()
            y_pred.extend(preds)
            y_true.extend(labels.tolist())

    val_f1 = compute_macro_f1(y_true, y_pred)
    val_acc = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f} | Waktu: {time.time()-t0:.1f}s")

    # Simpan model terbaik ke submission/model.pth
    if val_f1 >= best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), SUBMISSION_DIR / "model.pth")
        print(f"  --> Model disimpan di submission/model.pth (Macro-F1: {best_f1:.4f})")

# Copy backup ke Google Drive
drive_output_dir = Path("/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa/outputs")
drive_output_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), drive_output_dir / "model.pth")
print(f"\n✅ Training selesai! Cadangan model disalin ke Drive: {drive_output_dir / 'model.pth'}")

Perangkat yang digunakan: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 206MB/s]



--- Memulai Training di CUDA ---
Epoch 01/10 | Train Loss: 0.6739 | Val Acc: 0.8341 | Val Macro-F1: 0.8320 | Waktu: 1042.7s
  --> Model disimpan di submission/model.pth (Macro-F1: 0.8320)
Epoch 02/10 | Train Loss: 0.3716 | Val Acc: 0.8053 | Val Macro-F1: 0.8057 | Waktu: 42.2s
Epoch 03/10 | Train Loss: 0.3559 | Val Acc: 0.8341 | Val Macro-F1: 0.8326 | Waktu: 41.8s
  --> Model disimpan di submission/model.pth (Macro-F1: 0.8326)
Epoch 04/10 | Train Loss: 0.2395 | Val Acc: 0.8486 | Val Macro-F1: 0.8409 | Waktu: 42.4s
  --> Model disimpan di submission/model.pth (Macro-F1: 0.8409)
Epoch 05/10 | Train Loss: 0.2186 | Val Acc: 0.8654 | Val Macro-F1: 0.8626 | Waktu: 42.2s
  --> Model disimpan di submission/model.pth (Macro-F1: 0.8626)
Epoch 06/10 | Train Loss: 0.1676 | Val Acc: 0.8558 | Val Macro-F1: 0.8505 | Waktu: 42.6s
Epoch 07/10 | Train Loss: 0.1480 | Val Acc: 0.8558 | Val Macro-F1: 0.8552 | Waktu: 42.5s
Epoch 08/10 | Train Loss: 0.1662 | Val Acc: 0.8101 | Val Macro-F1: 0.8036 | Waktu: 42

In [16]:
import copy, shutil, subprocess, sys, time
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

# ---------- PENGATURAN ----------
BASE       = Path("/content/drive/MyDrive/Studio2_Kelompok11/dataset_mahasiswa")
VAL_DIR    = BASE / "local_split" / "val"
MODEL_PATH = Path("submission/model.pth")
EVAL_PY    = BASE / "starter_kit" / "evaluate.py"
CLASSES    = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
EXTS       = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
MEAN, STD  = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
K          = len(CLASSES)

# ---------- MUAT MODEL (bobot terbaik di submission/model.pth) ----------
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, K)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=True), strict=True)
model.to(DEVICE).eval()

# ---------- MUAT DATA VALIDASI ----------
items = [(p, ci) for ci, c in enumerate(CLASSES)
         for p in sorted((VAL_DIR / c).iterdir()) if p.suffix.lower() in EXTS]
images = []
for p, _ in items:
    im = Image.open(p).convert("RGB")
    im.load()
    images.append(im)
y_true = [c for _, c in items]
n_per_class = [y_true.count(i) for i in range(K)]
print(f"Gambar validasi: {len(items)} | per kelas: {dict(zip(CLASSES, n_per_class))} | device: {DEVICE}")


def make_tf(res):
    # res=224 sama persis dengan get_transform() di submission (Resize 256 -> CenterCrop 224)
    return transforms.Compose([
        transforms.Resize(round(res * 256 / 224)),
        transforms.CenterCrop(res),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


@torch.no_grad()
def predict(res, tta):
    tf, preds = make_tf(res), []
    for i in range(0, len(images), 32):
        x = torch.stack([tf(im) for im in images[i:i + 32]]).to(DEVICE)
        p = model(x).softmax(1)
        if tta:
            p = (p + model(torch.flip(x, dims=[3])).softmax(1)) / 2
        preds += p.argmax(1).cpu().tolist()
    return preds


def metrics(yt, yp):
    cm = [[0] * K for _ in range(K)]
    for t, p in zip(yt, yp):
        cm[t][p] += 1
    f1s = []
    for c in range(K):
        tp = cm[c][c]
        fp = sum(cm[r][c] for r in range(K)) - tp
        fn = sum(cm[c]) - tp
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / K, f1s, cm


def fmt(x):  # format angka gaya README (koma desimal)
    return f"{x:.4f}".replace(".", ",")


# ---------- 1. RESOLUSI x TTA ----------
print("\n" + "=" * 60 + "\n1. RESOLUSI x TTA (Macro-F1)\n" + "=" * 60)
res_table = {}
for res in (224, 256, 288):
    res_table[res] = (metrics(y_true, predict(res, False))[0],
                      metrics(y_true, predict(res, True))[0])
print("| **Resolusi** | **Tanpa TTA** | **TTA flip** |")
print("| ------------ | ------------- | ------------ |")
for res, (a, b) in res_table.items():
    print(f"| {res}          | {fmt(a)}        | {fmt(b)}       |")

# ---------- 2. DETAIL PER KELAS (224 px; tanpa TTA = setting submission saat ini) ----------
for tta in (False, True):
    yp = predict(224, tta)
    mf1, f1s, cm = metrics(y_true, yp)
    print("\n" + "=" * 60 + f"\n2. DETAIL 224 px | TTA flip = {tta}\n" + "=" * 60)
    print("| **Kelas**     | **F1**     | **Jumlah gambar** |")
    print("| ------------- | ---------- | ----------------- |")
    for c, f, n in zip(CLASSES, f1s, n_per_class):
        print(f"| {c:<13} | {fmt(f)}     | {n}                |")
    print(f"| **Macro-F1**  | **{fmt(mf1)}** |                   |")
    acc = sum(cm[i][i] for i in range(K)) / len(y_true)
    print(f"Akurasi: {fmt(acc)}")
    print("Confusion matrix (baris = label asli, kolom = prediksi):")
    for c, row in zip(CLASSES, cm):
        print(f"  {c:<15} {row}")
    errs = sorted(((cm[i][j], CLASSES[i], CLASSES[j]) for i in range(K) for j in range(K)
                   if i != j and cm[i][j]), reverse=True)
    print("Kesalahan terbanyak (asli -> prediksi):",
          ", ".join(f"{a} -> {b} ({n} gambar)" for n, a, b in errs[:5]) or "tidak ada")
    print("Kelas terlemah:", CLASSES[f1s.index(min(f1s))], f"(F1 {fmt(min(f1s))})")

# ---------- 3. WAKTU INFERENSI CPU 2 CORE ----------
print("\n" + "=" * 60 + "\n3. WAKTU INFERENSI (CPU, 2 thread)\n" + "=" * 60)
torch.set_num_threads(2)
cpu_model = copy.deepcopy(model).cpu().eval()
tf224, n = make_tf(224), min(50, len(images))
with torch.no_grad():
    cpu_model(tf224(images[0]).unsqueeze(0))  # warm-up
    t_pre = t_fwd = 0.0
    for im in images[:n]:
        t0 = time.perf_counter(); x = tf224(im).unsqueeze(0)
        t1 = time.perf_counter(); cpu_model(x)
        t2 = time.perf_counter()
        t_pre += t1 - t0; t_fwd += t2 - t1
pre_ms, fwd_ms = t_pre / n * 1000, t_fwd / n * 1000
print(f"Praproses ± {pre_ms:.0f} ms, forward ± {fwd_ms:.0f} ms per gambar (rata-rata {n} gambar)")
print(f"Tanpa TTA: ± {(pre_ms + fwd_ms) * 740 / 1000 / 60:.1f} menit untuk 740 gambar (batas 15 menit)")
print(f"Dengan TTA flip: ± {(pre_ms + 2 * fwd_ms) * 740 / 1000 / 60:.1f} menit untuk 740 gambar")
print("(CPU Colab tidak identik dengan komputer asisten; ini perkiraan kasar.)")

# ---------- 4. evaluate.py RESMI ----------
print("\n" + "=" * 60 + "\n4. evaluate.py RESMI (hasil sama seperti yang dipakai asisten)\n" + "=" * 60)
if EVAL_PY.exists() and Path("submission/model.py").exists():
    shutil.copy(EVAL_PY, "evaluate.py")
    r = subprocess.run([sys.executable, "evaluate.py", "--submission", "./submission",
                        "--data", str(VAL_DIR)], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print("evaluate.py atau submission/model.py tidak ditemukan. Cek path EVAL_PY.")

Gambar validasi: 416 | per kelas: {'botol_plastik': 78, 'gelas_plastik': 81, 'kertas_kardus': 55, 'saset_kemasan': 101, 'styrofoam': 101} | device: cuda

1. RESOLUSI x TTA (Macro-F1)
| **Resolusi** | **Tanpa TTA** | **TTA flip** |
| ------------ | ------------- | ------------ |
| 224          | 0,8626        | 0,8661       |
| 256          | 0,8535        | 0,8625       |
| 288          | 0,8475        | 0,8512       |

2. DETAIL 224 px | TTA flip = False
| **Kelas**     | **F1**     | **Jumlah gambar** |
| ------------- | ---------- | ----------------- |
| botol_plastik | 0,8630     | 78                |
| gelas_plastik | 0,8284     | 81                |
| kertas_kardus | 0,8522     | 55                |
| saset_kemasan | 0,8646     | 101                |
| styrofoam     | 0,9048     | 101                |
| **Macro-F1**  | **0,8626** |                   |
Akurasi: 0,8654
Confusion matrix (baris = label asli, kolom = prediksi):
  botol_plastik   [63, 6, 1, 3, 5]
  gelas_plastik   [2, 